# 05. Multi-step generation and long-term signals (stage 6)

Task question 4 and plan §8 (`docs/codeagents/research_plan.md`): what happens when the recommender generates five tracks in a row, each appended to the context as if the user had heard it. Run `04_rl` first: the policies are read from `runs/` on Drive.

`recsys_rl.generate.rollout_eval` runs every policy in three feedback modes for the appended listen (the model reads feedback tokens and has no "unknown" one):

- `accept`: played to the end, no like — every recommendation taken as consumed (plan mode (a));
- `expected`: the proxy RM's expected played ratio (mode (b));
- `sampled`: skip / partial / Listen+ and like / dislike drawn from the proxy RM (mode (c), a stochastic user simulator);

and two decodings: `greedy` (repeats inside the slate are not masked, so loops show up) and `sample`. Per step: relevance of the step's top-100 against the real Listen+ of the test day, KL to π_ref on the same extended context, entropy, what the generated track is (heard before, already in the slate, length, popularity, distinct artists in the slate) and the proxy / gold RM view of it with the ensemble spread (does the RM extrapolate more with every step?). Step 1 is the ordinary test evaluation.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import torch

from recsys_rl.config import load_config
from recsys_rl.data import prepare
from recsys_rl.generate import rollout_eval
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml")
prepared = prepare(cfg)
RESULTS = cfg.results_dir / "stage6"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)
runs = cfg.run_dir.parent

## Rollouts

Policies span the KL axis of stage 5: π_ref, RLOO R1 β = 0.1 (KL 0.14, the best RL point), RLOO R1 β = 0.02 (KL 1.3), GRPO β = 0.1 (KL 6, strong hacking: short tracks, low entropy) the distilled policy `distill` β = 0.05 (the best single model, observations §9) and the reranker R1 β = 0.1 recomputed on the extended context at every step (the RM sees the generated history too). Per-user tables are cached on Drive per policy (`runs/multistep/`), so a restarted session or a new policy only computes what is missing.

In [ ]:
POLICIES = {
    "pi_ref": None,
    "rl_ratio_b0.1": runs / "rl_ratio_b0.1" / "policy.pt",
    "rl_ratio_b0.02": runs / "rl_ratio_b0.02" / "policy.pt",
    "rl_ratio_grpo_b0.1": runs / "rl_ratio_grpo_b0.1" / "policy.pt",
    "rl_distill_b0.05": runs / "rl_distill_b0.05" / "policy.pt",  # the best single policy
}
RERANK = {"rr_ratio_b0.1": 0.1}
STEPS = 5
cache_dir = runs / "multistep"
cache_dir.mkdir(parents=True, exist_ok=True)
specs = {
    **{name: {"policies": {name: path}} for name, path in POLICIES.items()},
    **{name: {"rerank": {name: beta}} for name, beta in RERANK.items()},
}
parts = []
for name, spec in specs.items():  # one cache file per policy: new policies add to finished ones
    cache = cache_dir / f"rollouts_test_{name}.parquet"
    if not cache.exists():
        rollout_eval(cfg, prepared, device, split="test", steps=STEPS, **spec).write_parquet(cache)
    parts.append(pl.read_parquet(cache))
rollouts = pl.concat(parts).with_columns(
    gold_minus_proxy=pl.col("gold_ratio") - pl.col("proxy_ratio")
)
summary = (
    rollouts.group_by("policy", "mode", "decoding", "step", maintain_order=True)
    .agg(pl.exclude("user").cast(pl.Float64).mean())
    .sort("policy", "mode", "decoding", "step")
)
summary.write_csv(RESULTS / "rollouts_summary.csv")
summary.filter(pl.col("step").is_in([1, STEPS])).select(
    "policy",
    "mode",
    "decoding",
    "step",
    "ndcg@10",
    "hit",
    "kl_ref",
    "entropy",
    "in_history",
    "in_slate",
    "length",
    "slate_artists",
    "proxy_ratio",
    "gold_ratio",
    "gold_ratio_std",
)

## Metrics by step

One figure per decoding; colour = policy, line style = feedback mode. Step 1 is the same in all modes (nothing appended yet).

In [ ]:
PANELS = [
    ("ndcg@10", "NDCG@10 of the step's top-100 (real test Listen+)"),
    ("recall@100", "Recall@100"),
    ("hit", "generated track in the test Listen+"),
    ("kl_ref", "KL(π || π_ref) on the extended context"),
    ("entropy", "entropy"),
    ("in_history", "generated track heard before"),
    ("in_slate", "generated track already in the slate"),
    ("slate_artists", "distinct artists in the slate so far"),
    ("length", "length of the generated track, s"),
    ("log_count", "log10 train count of the generated track"),
    ("proxy_ratio", "proxy E[ratio] of the generated track"),
    ("gold_ratio", "gold E[ratio]"),
    ("gold_minus_proxy", "gold − proxy E[ratio]"),
    ("proxy_ratio_std", "proxy ensemble std"),
    ("gold_ratio_std", "gold ensemble std"),
]
STYLES = {"accept": "-", "expected": "--", "sampled": ":"}


def plot_steps(decoding: str) -> None:
    data = summary.filter(pl.col("decoding") == decoding)
    panels = [(col, title) for col, title in PANELS if col in data.columns]
    fig, axes = plt.subplots(3, 5, figsize=(26, 13))
    colours = {p: f"C{i}" for i, p in enumerate(data["policy"].unique(maintain_order=True))}
    for ax, (col, title) in zip(axes.ravel(), panels, strict=False):
        for (policy, mode), rows in data.group_by("policy", "mode", maintain_order=True):
            ax.plot(
                rows["step"],
                rows[col],
                STYLES[mode],
                marker="o",
                color=colours[policy],
                label=f"{policy} / {mode}",
            )
        ax.set(xlabel="step", title=title)
    for ax in axes.ravel()[len(panels) :]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(f"decoding: {decoding}")
    save_figure(fig, RESULTS / f"rollouts_{decoding}.png")
    plt.show()


for decoding in ("greedy", "sample"):
    plot_steps(decoding)

## Long-term signals (stage 6.2)

Task question 3 and plan §7.1. R6 rewards the RM head `continue`: p(the next listen starts within 30 min of this one). In stage 3 it was barely better than the user's own train mean (AUC 0.64 vs 0.63, observations §4), so before optimising it we check offline what it is a proxy *for*. `recsys_rl.longterm` relates the predictions on the real val / test listens to outcomes in the logs:

- `continue` (the label itself), the number of later listens in the session and the minutes until its last listen (sessions that may run past the cutoff are dropped), and for the val day whether the user comes back on the test day (`returns`);
- pooled Spearman / AUC and the **within-user** correlation (both sides minus the user's mean). For a fixed context a policy only changes the within-user part, so a signal that only separates active users from inactive ones is useless as a reward;
- user level: the mean prediction of the day vs returning the next day, against the number of listens of the day (activity) as a baseline;
- by track length: `continue` is counted from the start of the listen, so a long track cannot "continue" within 30 min — a likely cheap path for R6.

All of this is observational: it says whether the proxy *goes with* longer sessions, not whether recommending such tracks *causes* them.

In [ ]:
from recsys_rl import longterm, rm_eval
from recsys_rl.reward_model import HEAD_INDEX, build_feedback_set, load_reward_model, predict
from recsys_rl.train import amp_dtype

store = prepared.store.in_memory()
dtype = amp_dtype(device, cfg.rm.amp)
rms = {name: load_reward_model(runs / name / "best.pt", device) for name in ("rm_proxy", "rm_gold")}
h_cont = HEAD_INDEX["continue"]
validity, user_returns, by_length, deciles = [], [], [], []
for split in ("val", "test"):
    fs = build_feedback_set(
        store, prepared.bounds, split, cfg.data.max_seq_len, cfg.data, cfg.rm.skip_threshold
    )
    preds = {name: predict(rm, store, fs, amp_dtype=dtype).mean(0) for name, rm in rms.items()}
    baselines = rm_eval.baseline_predictions(
        store, prepared.bounds, fs, cfg.data, cfg.rm.skip_threshold
    )
    preds |= {name: baselines[name] for name in ("user_mean", "item_mean")}
    outcomes = longterm.session_outcomes(
        store,
        fs,
        cfg.data.session_gap_minutes,
        return_from=prepared.bounds.test_start if split == "val" else None,  # test has no next day
    )
    validity.append(longterm.validity_table(preds, fs, outcomes))
    by_length.append(longterm.length_table(preds, fs, outcomes))
    for within in (False, True):
        deciles.append(
            longterm.decile_table(
                preds["rm_proxy"][:, h_cont], fs.valid[:, h_cont], outcomes, within
            ).with_columns(split=pl.lit(split))
        )
    if split == "val":
        user_returns.append(longterm.user_return_table(preds, fs, outcomes))
validity, by_length = pl.concat(validity), pl.concat(by_length)
deciles, user_returns = pl.concat(deciles), pl.concat(user_returns)
validity.write_csv(RESULTS / "longterm_validity.csv")
by_length.write_csv(RESULTS / "longterm_length.csv")
deciles.write_csv(RESULTS / "longterm_deciles.csv")
user_returns.write_csv(RESULTS / "longterm_return.csv")
display(user_returns)
display(by_length)
validity.filter(pl.col("method").is_in(["rm_proxy", "rm_gold", "user_mean"])).sort(
    "split", "outcome", "method", "head"
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(
    axes, ["continue_rate", "remaining_listens", "remaining_minutes", "return_rate"], strict=True
):
    for (split, within), rows in deciles.group_by("split", "within_user", maintain_order=True):
        if rows[col].is_null().all():
            continue
        style = "--" if within else "-"
        label = f"{split}, {'within user' if within else 'pooled'}"
        ax.plot(rows["decile"] + 1, rows[col], style, marker="o", label=label)
    ax.set(xlabel="decile of proxy p(continue)", title=col)
axes[0].legend()
save_figure(fig, RESULTS / "longterm_deciles.png")
plt.show()

### Optimising R6

R6 at a KL comparable with the R1 points of stage 5: β is matched through the in-group reward std under π_ref (`results/stage4/reward_scales.csv`, as in step 5.2: β_R6 = β_R1 · σ_continue / σ_ratio). The reranker R6 (no training), RLOO R6 and `distill` R6 (the best way to amortise the reranker, observations §9) at the match of R1 β = 0.05; the reranker also at the match of β = 0.1. The question is what the policy changes to raise p(continue): if it is mostly shorter tracks, R6 is R1 with a different label, not a long-term goal.

In [ ]:
import math

from recsys_rl.config import parse_overrides
from recsys_rl.rerank import rerank_eval
from recsys_rl.rl import train_rl

scales = pl.read_csv(cfg.results_dir / "stage4" / "reward_scales.csv")
sigma = dict(zip(scales["term"], scales["group_std"], strict=True))
R6_BETA = {b: round(b * sigma["continue"] / sigma["ratio"], 5) for b in (0.1, 0.05)}
print("R6 beta matched to R1 beta:", R6_BETA)
EARLY = ["rl.eval_early_every=50", "rl.eval_early_until=500"]
R6_RUNS = {
    "rl_continue_eq0.05": ["rl.reward=continue", f"rl.kl_beta={R6_BETA[0.05]}", *EARLY],
    "rl_distill_continue_eq0.05": [
        "rl.method=distill",
        "rl.reward=continue",
        f"rl.kl_beta={R6_BETA[0.05]}",
        *EARLY,
    ],
}
for name, overrides in R6_RUNS.items():
    run_cfg = load_config("configs/base.yaml", **parse_overrides([f"run_name={name}", *overrides]))
    train_rl(run_cfg, prepared, device)
rerank_r6 = rerank_eval(
    load_config("configs/base.yaml", **parse_overrides(["rl.reward=continue"])),
    prepared,
    device,
    betas=(math.inf, R6_BETA[0.1], R6_BETA[0.05]),
    name="rerank_continue",
)
rerank_r6.write_csv(RESULTS / "rerank_continue_val.csv")
R6_PANELS = [
    ("val_proxy_reward@10", "proxy p(continue)@10"),
    ("val_gold_reward@10", "gold p(continue)@10"),
    ("val_ndcg@10", "NDCG@10"),
    ("val_kl_ref", "KL(π || π_ref)"),
    ("val_gold_ratio@10", "gold E[ratio]@10"),
    ("val_length@10", "mean track length@10, s"),
    ("val_repeat@10", "share of history tracks@10"),
    ("val_log_count@10", "mean log10 train count@10"),
]
r6_logs = pl.concat(
    [pl.read_ndjson(runs / name / "log.jsonl").with_columns(run=pl.lit(name)) for name in R6_RUNS]
    + [rerank_r6],
    how="diagonal_relaxed",
)
r6_logs.write_csv(RESULTS / "r6.csv")
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, (col, title) in zip(axes.ravel(), R6_PANELS, strict=True):
    for name, rows in r6_logs.group_by("run", maintain_order=True):
        ax.plot(rows["step"], rows[col], marker="o", label=name[0])
    ax.set(xlabel="step (reranker: β index, π_ref first)", title=title)
axes[0, 0].legend()
save_figure(fig, RESULTS / "r6.png")
plt.show()
rerank_r6.select(
    "beta",
    "val_ndcg@10",
    "val_kl_ref",
    "val_proxy_reward@10",
    "val_gold_reward@10",
    "val_gold_ratio@10",
    "val_length@10",
    "val_repeat@10",
)

### R6 vs R1: paired bootstrap

The same per-user comparison as in `04_rl` (`recsys_rl.significance`), with p(continue) of the top-10 under both RMs: R6 policies against π_ref and against their R1 counterparts (RLOO β = 0.05, `distill` β = 0.05, reranker β = 0.1 / 0.05). If R6 raises gold p(continue) only together with shorter tracks (`length@10`), it found the length path.

In [ ]:
from recsys_rl.significance import bootstrap_table, collect_per_user

R6_RERANK = {
    "rr_ratio_b0.1": ("ratio", 0.1),
    "rr_ratio_b0.05": ("ratio", 0.05),
    "rr_continue_eq0.1": ("continue", R6_BETA[0.1]),
    "rr_continue_eq0.05": ("continue", R6_BETA[0.05]),
}
R6_POLICIES = ["rl_ratio_b0.05", "rl_distill_b0.05", *R6_RUNS]
sig_dir = runs / "significance"
sig_dir.mkdir(parents=True, exist_ok=True)
per_user_r6 = []
for split in ("val", "test"):
    path = sig_dir / f"per_user_longterm_{split}.parquet"
    if not path.exists():
        collect_per_user(
            cfg,
            prepared,
            device,
            split=split,
            policies={name: runs / name / "policy.pt" for name in R6_POLICIES},
            rerank=R6_RERANK,
        ).write_parquet(path)
    per_user_r6.append(pl.read_parquet(path))
per_user_r6 = pl.concat(per_user_r6, how="diagonal_relaxed")
R6_PAIRS = [(m, "pi_ref") for m in [*R6_RUNS, "rr_continue_eq0.1", "rr_continue_eq0.05"]] + [
    ("rl_continue_eq0.05", "rl_ratio_b0.05"),
    ("rl_distill_continue_eq0.05", "rl_distill_b0.05"),
    ("rr_continue_eq0.1", "rr_ratio_b0.1"),
    ("rr_continue_eq0.05", "rr_ratio_b0.05"),
]
R6_METRICS = [
    "ndcg@10", "recall@100", "kl_ref", "repeat@10", "length@10", "log_count@10", "npcount@10",
    "proxy_continue@10", "gold_continue@10", "gold_ratio@10", "gold_debiased@10", "gold_like@10",
]  # fmt: skip
boot_r6 = bootstrap_table(per_user_r6, R6_PAIRS, R6_METRICS, n_boot=10_000, seed=cfg.seed)
boot_r6.write_csv(RESULTS / "bootstrap_longterm.csv")
boot_r6.filter(
    pl.col("metric").is_in(["ndcg@10", "gold_continue@10", "gold_ratio@10", "length@10"])
    & ~pl.col("exclude_seen")
).select(
    "split", "method", "baseline", "metric", "value", "baseline_value", "diff", "ci_low",
    "ci_high", "p_value", "p_holm",
)  # fmt: skip